# 🧬 Classification des cancers à partir de données d'expression génique (RNA-seq)

### Projet de Data Science & Bioinformatique

**Objectif :** Développer un modèle de Machine Learning capable de prédire le type de cancer d'un échantillon à partir de son profil d'expression génique.

---

## 📋 Sommaire

- [🎯 1. Objectif du projet](#-1-objectif-du-projet)
- [📦 2. Import des bibliothèques](#-2-import-des-bibliothèques)
- [📂 3. Chargement des données](#-3-chargement-des-données)
- [🔍 4. Compréhension du jeu de données](#-4-compréhension-du-jeu-de-données)
- [🧹 5. Nettoyage des données](#-5-nettoyage-des-données)
- [📊 6. Analyse exploratoire (EDA)](#-6-analyse-exploratoire-eda)
- [⚙️ 7. Préparation des données](#-7-préparation-des-données)
- [✂️ 8. Séparation Train / Test](#-8-séparation-train--test)
- [🌲 9. Premier modèle : Random Forest](#-9-premier-modèle--random-forest)
- [📈 10. Évaluation du modèle](#-10-évaluation-du-modèle)
- [📌 11. Conclusion et perspectives](#-11-conclusion-et-perspectives)

---

**Dataset utilisé :**

- **Nom :** TCGA Pan-Cancer RNA-seq Gene Expression
- **Nombre d'échantillons :** 801
- **Nombre de gènes :** 20 531
- **Nombre de classes :** 5 types de cancers

---

> 💡 **Objectif pédagogique :**
> Ce notebook documente toutes les étapes d'un projet de Machine Learning, depuis le chargement des données jusqu'à l'évaluation d'un premier modèle de classification.

---

# 🎯 1. Objectif du projet

L'objectif de ce projet est de construire un modèle de classification capable d'identifier le type de cancer d'un patient à partir de son profil d'expression génique (RNA-seq).

Le projet suit les principales étapes d'un workflow de Data Science :

1. Charger les données.
2. Comprendre leur structure.
3. Nettoyer les données.
4. Réaliser une analyse exploratoire (EDA).
5. Préparer les données pour le Machine Learning.
6. Entraîner un premier modèle de classification.
7. Évaluer les performances du modèle.

# 📦 2. Import des bibliothèques

Dans cette section, on importe les bibliothèques nécessaires pour :

- manipuler les données avec `pandas` ;
- effectuer des calculs numériques avec `numpy` ;
- créer des visualisations avec `matplotlib` ;
- préparer les données et entraîner un modèle avec `scikit-learn`.

In [8]:
# ==========================================
# Import des bibliothèques nécessaires
# ==========================================

# Manipulation des données
import pandas as pd
import numpy as np

# Visualisation des données
import matplotlib.pyplot as plt

# Style des graphiques
plt.style.use("ggplot")

# 📂 3. Chargement des données

Dans cette étape, nous chargeons les deux fichiers du jeu de données :

- **data.csv** : contient les niveaux d'expression de plus de 20 000 gènes pour chaque patient.
- **labels.csv** : contient le type de cancer associé à chaque patient.

L'objectif est de vérifier que les données sont correctement chargées avant de commencer leur exploration.

In [31]:
# ==========================================
# Chargement des données
# ==========================================

# Chargement des données d'expression génique
data = pd.read_csv("../data/TCGA-PANCAN-HiSeq-801x20531/data.csv")

# Chargement des labels (types de cancers)
labels = pd.read_csv("../data/TCGA-PANCAN-HiSeq-801x20531/labels.csv")

In [10]:
# Vérification des dimensions des jeux de données
print(f"Data shape   : {data.shape}")
print(f"Labels shape : {labels.shape}")

Data shape   : (801, 20532)
Labels shape : (801, 2)


## Interprétation des dimensions

Le fichier `data.csv` contient les variables explicatives du projet.

Chaque ligne correspond à un patient ou échantillon biologique.

Chaque colonne correspond soit à un identifiant, soit à un gène.

Le fichier `labels.csv` contient la variable cible, c'est-à-dire le type de cancer associé à chaque patient.

In [12]:
# Affichage des premières lignes du fichier data
data.head()

,Unnamed: 0,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
0,sample_0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,...,4.926711,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.0
1,sample_1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,...,4.593372,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.0
2,sample_2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,...,5.125213,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.0
3,sample_3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,...,6.076566,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.0
4,sample_4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,...,5.996032,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.0


In [13]:
# Affichage des premières lignes du fichier labels
labels.head()

,Unnamed: 0,Class
0,sample_0,PRAD
1,sample_1,LUAD
2,sample_2,PRAD
3,sample_3,PRAD
4,sample_4,BRCA


# 🔍 4. Compréhension du jeu de données

Avant de construire un modèle de Machine Learning, il est essentiel de comprendre la structure des données.

Nous allons répondre aux questions suivantes :

- Combien d'échantillons contient le jeu de données ?
- Combien de gènes sont mesurés ?
- Quel est le type de chaque variable ?
- Y a-t-il des valeurs manquantes ?
- Combien de classes de cancers sont présentes ?

In [14]:
# Dimensions des jeux de données

print(f"Nombre d'échantillons : {data.shape[0]}")
print(f"Nombre de colonnes : {data.shape[1]}")

print(f"\nNombre de labels : {labels.shape[0]}")

Nombre d'échantillons : 801
Nombre de colonnes : 20532

Nombre de labels : 801


## Interprétation

Le jeu de données contient :

- **801 patients** (ou échantillons biologiques).
- **20 532 colonnes** dans `data.csv` :
  - 1 colonne contenant l'identifiant de l'échantillon (`Unnamed: 0`) ;
  - 20 531 colonnes correspondant aux niveaux d'expression des gènes.
- **801 labels** contenant le type de cancer de chaque patient.

In [16]:
# Informations générales sur le DataFrame

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Columns: 20532 entries, Unnamed: 0 to gene_20530
dtypes: float64(20531), str(1)
memory usage: 125.5 MB


## Pourquoi utiliser `info()` ?

La méthode `info()` permet de connaître :

- le nombre de lignes ;
- le nombre de colonnes ;
- le type de chaque variable ;
- la quantité de valeurs non nulles.

Cela permet de détecter rapidement d'éventuels problèmes avant l'analyse.

In [21]:
# Vérification des valeurs manquantes
missing_values = data.isna().sum().sum()

print(f"Nombre total de valeurs manquantes : {missing_values}")

Nombre total de valeurs manquantes : 0


In [11]:
data.describe()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
count,801.000000,801.000000,801.000000,801.000000,801.000000,801.0,801.000000,801.000000,801.000000,801.000000,...,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000
mean,0.026642,3.010909,3.095350,6.722305,9.813612,0.0,7.405509,0.499882,0.016744,0.013428,...,5.896573,8.765891,10.056252,4.847727,9.741987,11.742228,10.155271,9.590726,5.528177,0.095411
std,0.136850,1.200828,1.065601,0.638819,0.506537,0.0,1.108237,0.508799,0.133635,0.204722,...,0.746399,0.603176,0.379278,2.382728,0.533898,0.670371,0.580569,0.563849,2.073859,0.364529
min,0.000000,0.000000,0.000000,5.009284,8.435999,0.0,3.930747,0.000000,0.000000,0.000000,...,2.853517,6.678368,8.669456,0.000000,7.974942,9.045255,7.530141,7.864533,0.593975,0.000000
25%,0.000000,2.299039,2.390365,6.303346,9.464466,0.0,6.676042,0.000000,0.000000,0.000000,...,5.454926,8.383834,9.826027,3.130750,9.400747,11.315857,9.836525,9.244219,4.092385,0.000000
50%,0.000000,3.143687,3.127006,6.655893,9.791599,0.0,7.450114,0.443076,0.000000,0.000000,...,5.972582,8.784144,10.066385,5.444935,9.784524,11.749802,10.191207,9.566511,5.218618,0.000000
75%,0.000000,3.883484,3.802534,7.038447,10.142324,0.0,8.121984,0.789354,0.000000,0.000000,...,6.411292,9.147136,10.299025,6.637412,10.082269,12.177852,10.578561,9.917888,6.876382,0.000000
max,1.482332,6.237034,6.063484,10.129528,11.355621,0.0,10.718190,2.779008,1.785592,4.067604,...,7.771054,11.105431,11.318243,9.207495,11.811632,13.715361,11.675653,12.813320,11.205836,5.254133


### Interprétation

Les statistiques descriptives montrent plusieurs points intéressants :

- Le jeu de données contient **801 échantillons** pour chacun des **20 531 gènes**.
- Certains gènes présentent une **expression nulle** chez de nombreux patients (minimum = 0).
- Les niveaux d'expression varient fortement d'un gène à l'autre.
- Certains gènes sont très peu variables tandis que d'autres présentent une forte dispersion.
- Aucune valeur manquante n'a été détectée, ce qui facilite les étapes de préparation des données.

À ce stade, rien n'indique d'anomalie évidente dans le jeu de données.

# 🧹 5. Nettoyage des données

Dans cette section, nous préparons les données avant l'analyse et le Machine Learning.

Les objectifs sont :

- renommer la colonne `Unnamed: 0` en `Sample` ;
- vérifier que les fichiers `data` et `labels` sont bien alignés ;
- fusionner les données d'expression génique avec les labels ;
- identifier les gènes constants ;
- supprimer les gènes qui n'apportent aucune information.

In [22]:
# ==========================================
# Renommage de la colonne d'identifiant
# ==========================================

# La colonne "Unnamed: 0" correspond à l'identifiant de chaque échantillon
data = data.rename(columns={"Unnamed: 0": "Sample"})
labels = labels.rename(columns={"Unnamed: 0": "Sample"})

# Vérification
data.head()

,Sample,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
0,sample_0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,...,4.926711,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.0
1,sample_1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,...,4.593372,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.0
2,sample_2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,...,5.125213,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.0
3,sample_3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,...,6.076566,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.0
4,sample_4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,...,5.996032,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.0


In [23]:
# Vérification des premières lignes du fichier labels
labels.head()

,Sample,Class
0,sample_0,PRAD
1,sample_1,LUAD
2,sample_2,PRAD
3,sample_3,PRAD
4,sample_4,BRCA


## Vérification de l'alignement des échantillons

Avant de fusionner les deux fichiers, il faut vérifier que les échantillons sont dans le même ordre dans `data` et `labels`.

C'est important car chaque ligne de `data` doit correspondre au bon type de cancer dans `labels`.

In [25]:
# Vérifier que les identifiants des échantillons sont identiques et dans le même ordre
same_samples = (data["Sample"] == labels["Sample"]).all()

print(f"Les échantillons sont-ils alignés ? {same_samples}")

Les échantillons sont-ils alignés ? True


Si le résultat est `True`, cela signifie que les deux fichiers sont correctement alignés.

On peut donc associer chaque profil d'expression génique au bon type de cancer.

In [26]:
# ==========================================
# Fusion des données avec les labels
# ==========================================

# Fusionner data et labels grâce à la colonne Sample
df = data.merge(labels, on="Sample")

# Vérification des dimensions
print(f"Dimensions du dataset fusionné : {df.shape}")

df.head()

Dimensions du dataset fusionné : (801, 20533)


,Sample,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,...,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530,Class
0,sample_0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,...,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.0,PRAD
1,sample_1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,...,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.0,LUAD
2,sample_2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,...,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.0,PRAD
3,sample_3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,...,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.0,PRAD
4,sample_4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,...,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.0,BRCA


## Dataset fusionné

Le nouveau DataFrame `df` contient maintenant :

- l'identifiant de l'échantillon (`Sample`) ;
- les niveaux d'expression des gènes ;
- la classe de cancer associée (`Class`).

Ce format est plus pratique pour l'exploration et la vérification des données.

In [28]:
# Vérification des colonnes principales
print("Première colonne :", df.columns[0])
print("Dernière colonne :", df.columns[-1])

Première colonne : Sample
Dernière colonne : Class


## Identification des gènes constants

Un gène constant est un gène qui possède la même valeur pour tous les patients.

Ce type de variable n'aide pas le modèle à distinguer les classes de cancer, car elle ne varie jamais.

Nous allons donc identifier ces gènes.

In [30]:
# Sélection des colonnes correspondant aux gènes
gene_columns = [col for col in df.columns if col.startswith("gene_")]

# Identification des gènes constants
constant_genes = df[gene_columns].columns[df[gene_columns].nunique() <= 1]

print(f"Nombre de gènes constants : {len(constant_genes)}")
print("Exemples de gènes constants :", list(constant_genes[:10]))

Nombre de gènes constants : 267
Exemples de gènes constants : ['gene_5', 'gene_23', 'gene_4370', 'gene_4808', 'gene_4809', 'gene_4814', 'gene_4816', 'gene_4817', 'gene_4831', 'gene_5288']


## Identification des gènes constants

Nous avons recherché les gènes dont la valeur est identique pour tous les patients.

Résultat :

- **267 gènes sont constants**.
- Ces gènes ne présentent aucune variation entre les échantillons.
- Ils n'apportent donc aucune information utile pour la classification des cancers.

Ces variables peuvent être supprimées sans perte d'information pour le modèle de Machine Learning.

> **Pourquoi supprimer les gènes constants ?**

Un algorithme de Machine Learning apprend en recherchant des différences entre les observations.

Si un gène possède toujours la même valeur, il ne permet pas de distinguer un patient d'un autre.

Le conserver augmente inutilement la dimension du jeu de données sans améliorer les performances du modèle.

In [32]:
df_clean = df.drop(columns=constant_genes)

print(df.shape)
print(df_clean.shape)

(801, 20533)
(801, 20266)


In [33]:
df_clean

,Sample,gene_0,gene_1,gene_2,gene_3,gene_4,gene_6,gene_7,gene_8,gene_9,...,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530,Class
0,sample_0,0.0,2.017209,3.265527,5.478487,10.431999,7.175175,0.591871,0.0,0.0,...,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.000000,PRAD
1,sample_1,0.0,0.592732,1.588421,7.586157,9.623011,6.816049,0.000000,0.0,0.0,...,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.000000,LUAD
2,sample_2,0.0,3.511759,4.327199,6.881787,9.870730,6.972130,0.452595,0.0,0.0,...,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.000000,PRAD
3,sample_3,0.0,3.663618,4.507649,6.659068,10.196184,7.843375,0.434882,0.0,0.0,...,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.000000,PRAD
4,sample_4,0.0,2.655741,2.821547,6.539454,9.738265,6.566967,0.360982,0.0,0.0,...,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.000000,BRCA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
796,sample_796,0.0,1.865642,2.718197,7.350099,10.006003,6.764792,0.496922,0.0,0.0,...,9.118313,10.004852,4.484415,9.614701,12.031267,9.813063,10.092770,8.819269,0.000000,BRCA
797,sample_797,0.0,3.942955,4.453807,6.346597,10.056868,7.320331,0.000000,0.0,0.0,...,9.623335,9.823921,6.555327,9.064002,11.633422,10.317266,8.745983,9.659081,0.000000,LUAD
798,sample_798,0.0,3.249582,3.707492,8.185901,9.504082,7.536589,1.811101,0.0,0.0,...,8.610704,10.485517,3.589763,9.350636,12.180944,10.681194,9.466711,4.677458,0.586693,COAD
799,sample_799,0.0,2.590339,2.787976,7.318624,9.987136,9.213464,0.000000,0.0,0.0,...,8.605387,11.004677,4.745888,9.626383,11.198279,10.335513,10.400581,5.718751,0.000000,PRAD


In [34]:
# Vérification des doublons
duplicates = df_clean.duplicated().sum()

print(f"Nombre de doublons : {duplicates}")

Nombre de doublons : 0


In [35]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Columns: 20266 entries, Sample to Class
dtypes: float64(20264), str(2)
memory usage: 123.8 MB


In [36]:
labels["Class"].value_counts()

Class
BRCA    300
KIRC    146
LUAD    141
PRAD    136
COAD     78
Name: count, dtype: int64

In [37]:
print(df_clean.shape)

(801, 20266)


In [38]:
df_clean

,Sample,gene_0,gene_1,gene_2,gene_3,gene_4,gene_6,gene_7,gene_8,gene_9,...,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530,Class
0,sample_0,0.0,2.017209,3.265527,5.478487,10.431999,7.175175,0.591871,0.0,0.0,...,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.000000,PRAD
1,sample_1,0.0,0.592732,1.588421,7.586157,9.623011,6.816049,0.000000,0.0,0.0,...,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.000000,LUAD
2,sample_2,0.0,3.511759,4.327199,6.881787,9.870730,6.972130,0.452595,0.0,0.0,...,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.000000,PRAD
3,sample_3,0.0,3.663618,4.507649,6.659068,10.196184,7.843375,0.434882,0.0,0.0,...,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.000000,PRAD
4,sample_4,0.0,2.655741,2.821547,6.539454,9.738265,6.566967,0.360982,0.0,0.0,...,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.000000,BRCA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
796,sample_796,0.0,1.865642,2.718197,7.350099,10.006003,6.764792,0.496922,0.0,0.0,...,9.118313,10.004852,4.484415,9.614701,12.031267,9.813063,10.092770,8.819269,0.000000,BRCA
797,sample_797,0.0,3.942955,4.453807,6.346597,10.056868,7.320331,0.000000,0.0,0.0,...,9.623335,9.823921,6.555327,9.064002,11.633422,10.317266,8.745983,9.659081,0.000000,LUAD
798,sample_798,0.0,3.249582,3.707492,8.185901,9.504082,7.536589,1.811101,0.0,0.0,...,8.610704,10.485517,3.589763,9.350636,12.180944,10.681194,9.466711,4.677458,0.586693,COAD
799,sample_799,0.0,2.590339,2.787976,7.318624,9.987136,9.213464,0.000000,0.0,0.0,...,8.605387,11.004677,4.745888,9.626383,11.198279,10.335513,10.400581,5.718751,0.000000,PRAD
